# 03 — Análise exploratória (Fase 5)
**Base:** CAT (predição + XAI) · o mesmo notebook roda para BC trocando `BASE`.

Seções: **1. Univariada** · **2. Bivariada (χ² + V de Cramér)** · 3. Informação mútua · 4. Séries temporais e UF · 5. Desbalanceamento dos alvos.

> Se já existe um `03_eda.ipynb`, copie estas células para dentro dele em vez de sobrescrever.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

from inss_acidentes.common.eda_univariada import (
    ConfigEDA, preparar_dataframe, classificar_colunas, executar_univariada,
    frequencias, plotar_pareto, plotar_histograma,
)
from inss_acidentes.common.eda_bivariada import (
    preparar_bivariada, calcular_pares, ranking_alvo, plotar_heatmap_v, plotar_associacao_alvo,
)

BASE = "cat"            # "cat" ou "beneficios"
RAIZ = Path.cwd().resolve()
while not (RAIZ / "configs").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent

cfg = ConfigEDA.from_yaml(RAIZ / "configs" / BASE / "eda.yaml", RAIZ)
df = preparar_dataframe(pd.read_parquet(cfg.entrada), cfg)     # converte datas/competências e texto -> category
print(f"{len(df):,} linhas × {df.shape[1]} colunas".replace(",", "."))

## 1. Univariada
### 1.1 Classificação das colunas
Corrija em `configs/<base>/eda.yaml` (`categoricas`, `numericas`, `datas`, `excluir`, `candidatos_alvo`).

In [ ]:
tipos = classificar_colunas(df, cfg)
tipos

### 1.2 Execução completa (grava tabelas e figuras em `reports/<base>/`)

In [ ]:
res = executar_univariada(df, cfg)
display(res['resumo'])

### 1.3 Cobertura de Pareto
Quantas categorias cobrem 80% das ocorrências — base para decidir agrupamentos.

In [ ]:
display(res['cobertura'].sort_values('n_categorias', ascending=False))

### 1.4 Pareto de uma variável (inline)

In [ ]:
col = tipos["categoricas"][0]        # troque pela variável de interesse
plotar_pareto(frequencias(df[col]), f"Pareto — {col}", cfg.top_n_pareto, cfg.limiar_pareto, df[col].isna().mean())

### 1.5 Numéricas

In [ ]:
display(res["estatisticas"].T)
for c in tipos["numericas"]:
    display(plotar_histograma(df[c], f"Distribuição — {c}", cfg.recortar_percentil))

## 2. Bivariada entre categóricas — χ² + V de Cramér (corrigido)
Métrica principal: **V de Cramér com correção de viés**. Com milhões de linhas o p-valor do χ² é ~0 para qualquer par; o que informa é o **tamanho do efeito** (V).
Numéricas entram em faixas fixas; datas entram como indicador de presença (detecta vazamento); níveis < 0,05% viram "Outras".

### 2.1 Quem entra e quem não entra

In [ ]:
df_b, codif, nao_avaliadas, params = preparar_bivariada(df, cfg)
print(f"{len(codif)} variáveis avaliadas")
display(nao_avaliadas)

### 2.2 Matriz de associação (todos os pares)

In [ ]:
pares, matriz = calcular_pares(codif)
display(pares.head(20)[["var_a", "var_b", "v_corrigido", "v_bruto", "n"]])
plotar_heatmap_v(matriz, params.anotar_heatmap_ate)

### 2.3 Pares redundantes / derivados (V ≥ limiar)
Não devem entrar juntos nos modelos (ou um deles sai).

In [ ]:
display(pares[pares['v_corrigido'] >= params.limiar_redundancia][['var_a', 'var_b', 'v_corrigido']])

### 2.4 Associação com cada candidato a alvo
Barras vermelhas: V ≥ limiar → derivada do alvo ou **possível vazamento** (excluir das features).

In [ ]:
for alvo in cfg.candidatos_alvo:
    if alvo in codif:
        display(plotar_associacao_alvo(ranking_alvo(pares, alvo, params.limiar_redundancia), alvo,
                                       params.top_n_alvo, params.limiar_redundancia))

### 2.5 Leitura e decisões (preencher)
| Variável / par | Observação | Decisão |
|---|---|---|
| | | |